# Feature Engineering

 What early signals (first 8 weeks) from trend data and customer reviews correlate with a style's seasonal peak popularity?

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR    = Path('data')
RESULTS_DIR = Path('results')

WINDOW   = 8
TS_COLS  = ['ts_mean', 'ts_slope', 'ts_curvature', 'ts_range', 'ts_peak_pos']
REV_COLS = ['rev_count', 'rev_mean_rating', 'rev_positive_frac']
TARGET   = 'actual_peak_value'

## Observation window

WINDOW = 8 means we use only the first 8 weeks of a season - exactly what a retailer has available when placing their first re-order. Everything after week 8 is the future we are trying to predict.

Why 8? It is a pragmatic trade-off: fewer weeks = earlier decision, but weaker signal.

In [5]:
trends  = pd.read_csv(DATA_DIR / 'trends_synthetic.csv', parse_dates=['date'])
reviews = pd.read_csv(DATA_DIR / 'reviews_synthetic.csv')
truth   = pd.read_csv(RESULTS_DIR / 'peak_ground_truth.csv')

STYLE_NAMES = truth['style'].unique().tolist()
N_YEARS     = int(truth['year'].nunique())
print(f'Styles: {len(STYLE_NAMES)}, Years: {N_YEARS}, Instances: {len(STYLE_NAMES) * N_YEARS}')

Styles: 8, Years: 6, Instances: 48


## Sample size

48 instances (8 styles × 6 years) - a small dataset for machine learning. This places real limits on what we can reliably detect:

- Walk-forward CV with 4 folds yields only 32 test instances (8 styles × 4 test years).
- A 1–2 pt difference in MAE may be within noise at this scale.

In [6]:
def ts_features(year_series: np.ndarray, window: int = WINDOW) -> dict:
    """Extract shape features from the first `window` weeks of a yearly series.
    """
    w = year_series[:window].astype(float)
    t = np.arange(window, dtype=float)
    slope, _   = np.polyfit(t, w, 1)
    a2, _, _   = np.polyfit(t, w, 2)
    return {
        'ts_mean':      float(np.mean(w)),
        'ts_slope':     float(slope),
        'ts_curvature': float(a2),
        'ts_range':     float(np.max(w) - np.min(w)),
        'ts_peak_pos':  int(np.argmax(w)),
    }


def rev_features(df_rev: pd.DataFrame, style: str, year: int, window: int = WINDOW) -> dict:
    """Extract review signals from the first `window` weeks of a season.
    """
    mask = ((df_rev['style'] == style) & (df_rev['year'] == year)
            & (df_rev['week_in_year'] <= window))
    sub  = df_rev.loc[mask]
    if len(sub) == 0:
        return {'rev_count': 0, 'rev_mean_rating': 0.0, 'rev_positive_frac': 0.0}
    return {
        'rev_count':         len(sub),
        'rev_mean_rating':   float(sub['rating'].mean()),
        'rev_positive_frac': float((sub['rating'] >= 4).mean()),
    }